# 03 - Run allocation v1

This notebook combines prepared demand/context data with prepared workforce capacity.
It creates the final MSOA/month/bucket allocation output.

In [16]:
import pandas as pd
from pathlib import Path
import sqlite3

## Settings

`RHO` controls the split between baseline and demand-based allocation.
For v1 it is set to 50/50.

In [17]:
DATA_DIR = Path("../../data")
OUTPUT_DIR = Path("../outputs")
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"

RHO = 0.5
RURALITY_WEIGHT = 0.1

UNCERTAINTY_IQR_RATIO_THRESHOLD = 1.25
UNCERTAINTY_CHANGE_PCT_THRESHOLD = 10
UNCERTAINTY_CHANGE_ABS_THRESHOLD = 2

bucket_baseline_shares = pd.DataFrame(
    [
        {"bucket": "local_reassurance", "bucket_baseline_share": 0.30},
        {"bucket": "acquisitive_crime", "bucket_baseline_share": 0.40},
        {"bucket": "disorder_damage", "bucket_baseline_share": 0.30},
    ]
)

## Load prepared tables

In [18]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_bucket_demand = pd.read_sql_query("SELECT * FROM msoa_bucket_demand_v1;", conn)
    msoa_context = pd.read_sql_query("SELECT * FROM msoa_context_v1;", conn)
    force_capacity_model = pd.read_sql_query("SELECT * FROM force_capacity_v1;", conn)
    capacity_settings = pd.read_sql_query("SELECT * FROM capacity_settings_v1;", conn)

capacity_version = capacity_settings.loc[0, "capacity_version"]

print("capacity settings:")
print(capacity_settings)
print("demand rows:", len(msoa_bucket_demand))
print("MSOA rows:", len(msoa_context))
print("capacity rows:", len(force_capacity_model))

capacity settings:
   workforce_year capacity_version     capacity_filter_description  \
0            2025   local_policing  Wider function: Local policing   

  wider_function_name  
0      local policing  
demand rows: 61704
MSOA rows: 6856
capacity rows: 43


## Baseline allocation

Baseline allocation depends on population, with a small rurality uplift.

In [19]:
msoa_context["baseline_score"] = (
    msoa_context["population"]
    * (1 + RURALITY_WEIGHT * msoa_context["rurality_score"])
)

force_baseline_totals = (
    msoa_context
    .groupby("pfa_code", as_index=False)
    .agg(force_baseline_score=("baseline_score", "sum"))
)

msoa_context = msoa_context.merge(force_baseline_totals, on="pfa_code", how="left")
msoa_context["baseline_share"] = (
    msoa_context["baseline_score"] / msoa_context["force_baseline_score"]
)

msoa_context = msoa_context.merge(
    force_capacity_model[
        [
            "pfa_code",
            "pcso_fte",
            "regular_police_staff_fte",
            "total_capacity",
        ]
    ],
    on="pfa_code",
    how="left",
)

msoa_context["capacity_base"] = RHO * msoa_context["total_capacity"]
msoa_context["capacity_var"] = (1 - RHO) * msoa_context["total_capacity"]
msoa_context["baseline_alloc_total"] = (
    msoa_context["capacity_base"] * msoa_context["baseline_share"]
)

msoa_context.head()

,msoa_code,msoa_name,pfa_code,pfa_name,population,lsoa_count,rurality_score,rural_lsoa_count,total_lsoa_count,baseline_score,force_baseline_score,baseline_share,pcso_fte,regular_police_staff_fte,total_capacity,capacity_base,capacity_var,baseline_alloc_total
0,E02000001,City of London 001,E23000034,"London, City of",11457,6,0.0,0,6,11457.0,11457.0,1.000000,3.0,204.0,207.0,103.5,103.5,103.500000
1,E02000002,Barking and Dagenham 001,E23000001,Metropolitan Police,8386,4,0.0,0,4,8386.0,8845554.0,0.000948,1263.0,12499.0,13762.0,6881.0,6881.0,6.523511
2,E02000003,Barking and Dagenham 002,E23000001,Metropolitan Police,11812,6,0.0,0,6,11812.0,8845554.0,0.001335,1263.0,12499.0,13762.0,6881.0,6881.0,9.188613
3,E02000004,Barking and Dagenham 003,E23000001,Metropolitan Police,6873,4,0.0,0,4,6873.0,8845554.0,0.000777,1263.0,12499.0,13762.0,6881.0,6881.0,5.346541
4,E02000005,Barking and Dagenham 004,E23000001,Metropolitan Police,11032,6,0.0,0,6,11032.0,8845554.0,0.001247,1263.0,12499.0,13762.0,6881.0,6881.0,8.581847


## Force resource summary

This table compares how many MSOAs each force covers with its available wider Local policing resources.


In [20]:
force_msoa_resource_summary = (
    msoa_context
    .groupby(["pfa_code", "pfa_name"], as_index=False)
    .agg(
        msoa_count=("msoa_code", "nunique"),
        lsoa_count=("lsoa_count", "sum"),
        population=("population", "sum"),
        total_capacity=("total_capacity", "first"),
        pcso_fte=("pcso_fte", "first"),
        regular_police_staff_fte=("regular_police_staff_fte", "first"),
    )
)

force_msoa_resource_summary["fte_per_msoa"] = (
    force_msoa_resource_summary["total_capacity"]
    / force_msoa_resource_summary["msoa_count"]
)

force_msoa_resource_summary["msoas_per_100_fte"] = (
    force_msoa_resource_summary["msoa_count"]
    / force_msoa_resource_summary["total_capacity"]
    * 100
)

force_msoa_resource_summary["population_per_fte"] = (
    force_msoa_resource_summary["population"]
    / force_msoa_resource_summary["total_capacity"]
)

force_msoa_resource_summary = force_msoa_resource_summary.sort_values(
    "fte_per_msoa",
    ascending=False,
).copy()

force_msoa_resource_summary.head(10)


,pfa_code,pfa_name,msoa_count,lsoa_count,population,total_capacity,pcso_fte,regular_police_staff_fte,fte_per_msoa,msoas_per_100_fte,population_per_fte
33,E23000034,"London, City of",1,6,11457,207.0,3.0,204.0,207.000000,0.483092,55.347826
0,E23000001,Metropolitan Police,1001,4980,8845554,13762.0,1263.0,12499.0,13.748252,7.273652,642.752071
4,E23000005,Greater Manchester,353,1688,2891156,3578.0,265.0,3313.0,10.135977,9.865847,808.036892
26,E23000027,Hertfordshire,154,713,1205137,1545.0,127.0,1418.0,10.032468,9.967638,780.023948
9,E23000010,West Yorkshire,301,1404,2376278,2961.0,476.0,2485.0,9.837209,10.165485,802.525498
6,E23000007,Northumbria,188,927,1465242,1824.0,78.0,1746.0,9.702128,10.307018,803.312500
7,E23000008,Durham,79,396,637117,766.0,122.0,644.0,9.696203,10.313316,831.745431
12,E23000013,Cleveland,75,359,579710,725.0,67.0,658.0,9.666667,10.344828,799.600000
3,E23000004,Merseyside,185,923,1441162,1776.0,165.0,1611.0,9.600000,10.416667,811.465090
13,E23000014,West Midlands,357,1719,2948513,3353.0,285.0,3068.0,9.392157,10.647182,879.365643


In [21]:
months = pd.DataFrame({"month": sorted(msoa_bucket_demand["month"].unique())})

msoa_month_context = msoa_context.merge(months, how="cross")

baseline_allocation = msoa_month_context.merge(bucket_baseline_shares, how="cross")
baseline_allocation["baseline_alloc"] = (
    baseline_allocation["baseline_alloc_total"]
    * baseline_allocation["bucket_baseline_share"]
)

baseline_allocation = baseline_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "baseline_alloc",
        "total_capacity",
        "capacity_base",
        "capacity_var",
    ]
].copy()

baseline_allocation.head()

,pfa_code,pfa_name,msoa_code,msoa_name,month,bucket,baseline_alloc,total_capacity,capacity_base,capacity_var
0,E23000034,"London, City of",E02000001,City of London 001,2025-12,local_reassurance,31.05,207.0,103.5,103.5
1,E23000034,"London, City of",E02000001,City of London 001,2025-12,acquisitive_crime,41.40,207.0,103.5,103.5
2,E23000034,"London, City of",E02000001,City of London 001,2025-12,disorder_damage,31.05,207.0,103.5,103.5
3,E23000034,"London, City of",E02000001,City of London 001,2026-01,local_reassurance,31.05,207.0,103.5,103.5
4,E23000034,"London, City of",E02000001,City of London 001,2026-01,acquisitive_crime,41.40,207.0,103.5,103.5


In [22]:
baseline_allocation.groupby(["pfa_name", "month"])["baseline_alloc"].sum()

pfa_name           month  
Avon and Somerset  2025-12     853.5
                   2026-01     853.5
                   2026-02     853.5
Bedfordshire       2025-12     261.5
                   2026-01     261.5
                               ...  
West Yorkshire     2026-01    1480.5
                   2026-02    1480.5
Wiltshire          2025-12     321.5
                   2026-01     321.5
                   2026-02     321.5
Name: baseline_alloc, Length: 117, dtype: float64

## Variable allocation

Variable allocation follows forecast demand within each force, month, and crime bucket.

In [23]:
force_bucket_demand = (
    msoa_bucket_demand
    .groupby(["pfa_code", "pfa_name", "month", "bucket"], as_index=False)
    .agg(force_bucket_demand=("weighted_demand", "sum"))
)

force_month_demand = (
    force_bucket_demand
    .groupby(["pfa_code", "pfa_name", "month"], as_index=False)
    .agg(force_total_demand=("force_bucket_demand", "sum"))
)

force_bucket_demand = force_bucket_demand.merge(
    force_month_demand,
    on=["pfa_code", "pfa_name", "month"],
    how="left",
)

force_bucket_demand["bucket_demand_share"] = (
    force_bucket_demand["force_bucket_demand"]
    / force_bucket_demand["force_total_demand"]
)

force_bucket_demand = force_bucket_demand.merge(
    force_capacity_model[["pfa_code", "total_capacity"]],
    on="pfa_code",
    how="left",
)

force_bucket_demand["capacity_var"] = (1 - RHO) * force_bucket_demand["total_capacity"]
force_bucket_demand["variable_bucket_capacity"] = (
    force_bucket_demand["capacity_var"]
    * force_bucket_demand["bucket_demand_share"]
)

force_bucket_demand.head()

,pfa_code,pfa_name,month,bucket,force_bucket_demand,force_total_demand,bucket_demand_share,total_capacity,capacity_var,variable_bucket_capacity
0,E23000001,Metropolitan Police,2025-12,acquisitive_crime,10148.39053,42666.19561,0.237856,13762.0,6881.0,1636.683895
1,E23000001,Metropolitan Police,2025-12,disorder_damage,27451.83828,42666.19561,0.643410,13762.0,6881.0,4427.301204
2,E23000001,Metropolitan Police,2025-12,local_reassurance,5065.96680,42666.19561,0.118735,13762.0,6881.0,817.014900
3,E23000001,Metropolitan Police,2026-01,acquisitive_crime,10146.24404,43549.23934,0.232983,13762.0,6881.0,1603.157858
4,E23000001,Metropolitan Police,2026-01,disorder_damage,27800.59412,43549.23934,0.638372,13762.0,6881.0,4392.634430


In [24]:
variable_allocation = msoa_bucket_demand.merge(
    force_bucket_demand[
        [
            "pfa_code",
            "pfa_name",
            "month",
            "bucket",
            "force_bucket_demand",
            "variable_bucket_capacity",
        ]
    ],
    on=["pfa_code", "pfa_name", "month", "bucket"],
    how="left",
)

variable_allocation["msoa_bucket_demand_share"] = (
    variable_allocation["weighted_demand"]
    / variable_allocation["force_bucket_demand"]
)

variable_allocation["variable_alloc"] = (
    variable_allocation["variable_bucket_capacity"]
    * variable_allocation["msoa_bucket_demand_share"]
)

variable_allocation = variable_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "demand_q25",
        "demand",
        "demand_q75",
        "weighted_demand_q25",
        "weighted_demand",
        "weighted_demand_q75",
        "force_bucket_demand",
        "variable_bucket_capacity",
        "msoa_bucket_demand_share",
        "variable_alloc",
    ]
].copy()

variable_allocation.head()

,pfa_code,pfa_name,msoa_code,msoa_name,month,bucket,demand_q25,demand,demand_q75,weighted_demand_q25,weighted_demand,weighted_demand_q75,force_bucket_demand,variable_bucket_capacity,msoa_bucket_demand_share,variable_alloc
0,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2025-12,acquisitive_crime,3.3986,9.6647,20.3897,1.359685,4.151725,9.559100,10148.39053,1636.683895,0.000409,0.669570
1,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2025-12,disorder_damage,19.3513,33.3510,51.4709,18.726220,30.996640,46.601080,27451.83828,4427.301204,0.001129,4.998990
2,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2025-12,local_reassurance,6.3696,11.9587,18.8288,2.866320,5.381415,8.472960,5065.96680,817.014900,0.001062,0.867889
3,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2026-01,acquisitive_crime,3.3632,9.6296,20.6521,1.345525,4.165125,9.697765,10146.24404,1603.157858,0.000411,0.658111
4,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2026-01,disorder_damage,19.5435,33.8425,52.4883,18.872920,31.458680,47.511020,27800.59412,4392.634430,0.001132,4.970630


In [25]:
variable_allocation.groupby(["pfa_name", "month"])["variable_alloc"].sum()

pfa_name           month  
Avon and Somerset  2025-12     853.5
                   2026-01     853.5
                   2026-02     853.5
Bedfordshire       2025-12     261.5
                   2026-01     261.5
                               ...  
West Yorkshire     2026-01    1480.5
                   2026-02    1480.5
Wiltshire          2025-12     321.5
                   2026-01     321.5
                   2026-02     321.5
Name: variable_alloc, Length: 117, dtype: float64

## Final allocation

In [26]:
final_allocation = baseline_allocation.merge(
    variable_allocation[
        [
            "pfa_code",
            "msoa_code",
            "month",
            "bucket",
            "demand_q25",
            "demand",
            "demand_q75",
            "weighted_demand_q25",
            "weighted_demand",
            "weighted_demand_q75",
            "variable_alloc",
        ]
    ],
    on=["pfa_code", "msoa_code", "month", "bucket"],
    how="left",
)

for col in ["demand_q25", "demand", "demand_q75", "weighted_demand_q25", "weighted_demand", "weighted_demand_q75"]:
    final_allocation[col] = final_allocation[col].fillna(0)

final_allocation["variable_alloc"] = final_allocation["variable_alloc"].fillna(0)

final_allocation["total_alloc"] = (
    final_allocation["baseline_alloc"]
    + final_allocation["variable_alloc"]
)

final_allocation["capacity_version"] = capacity_version

final_allocation["share_of_force_capacity"] = (
    final_allocation["total_alloc"]
    / final_allocation["total_capacity"]
)

final_allocation = final_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "capacity_version",
        "demand_q25",
        "demand",
        "demand_q75",
        "weighted_demand_q25",
        "weighted_demand",
        "weighted_demand_q75",
        "baseline_alloc",
        "variable_alloc",
        "total_alloc",
        "share_of_force_capacity",
        "total_capacity",
    ]
].copy()

final_allocation.head()

,pfa_code,pfa_name,msoa_code,msoa_name,month,bucket,capacity_version,demand_q25,demand,demand_q75,weighted_demand_q25,weighted_demand,weighted_demand_q75,baseline_alloc,variable_alloc,total_alloc,share_of_force_capacity,total_capacity
0,E23000034,"London, City of",E02000001,City of London 001,2025-12,local_reassurance,local_policing,3.6837,12.3167,24.2662,1.657665,5.542515,10.919790,31.05,1.330308,32.380308,0.156427,207.0
1,E23000034,"London, City of",E02000001,City of London 001,2025-12,acquisitive_crime,local_policing,412.1199,473.4940,541.1993,171.748980,198.981405,230.177670,41.40,47.759298,89.159298,0.430721,207.0
2,E23000034,"London, City of",E02000001,City of London 001,2025-12,disorder_damage,local_policing,202.4069,250.8728,303.3188,184.696840,226.692120,271.455960,31.05,54.410394,85.460394,0.412852,207.0
3,E23000034,"London, City of",E02000001,City of London 001,2026-01,local_reassurance,local_policing,6.5882,15.1189,26.8460,2.964690,6.803505,12.080700,31.05,1.637541,32.687541,0.157911,207.0
4,E23000034,"London, City of",E02000001,City of London 001,2026-01,acquisitive_crime,local_policing,412.6120,469.9859,541.7709,171.890975,197.513905,230.494895,41.40,47.539785,88.939785,0.429661,207.0


In [27]:
final_allocation = final_allocation.sort_values(
    ["pfa_code", "msoa_code", "bucket", "month"]
).copy()

final_allocation["prev_month_total_alloc"] = (
    final_allocation
    .groupby(["pfa_code", "msoa_code", "bucket"])["total_alloc"]
    .shift(1)
)

final_allocation["alloc_change_abs"] = (
    final_allocation["total_alloc"]
    - final_allocation["prev_month_total_alloc"]
)

final_allocation["alloc_change_pct"] = (
    final_allocation["alloc_change_abs"]
    / final_allocation["prev_month_total_alloc"]
    * 100
)

final_allocation["demand_iqr"] = (
    final_allocation["demand_q75"]
    - final_allocation["demand_q25"]
)

final_allocation["demand_iqr_ratio"] = (
    final_allocation["demand_iqr"]
    / final_allocation["demand"].clip(lower=1)
)

final_allocation["prev_month_demand"] = (
    final_allocation
    .groupby(["pfa_code", "msoa_code", "bucket"])["demand"]
    .shift(1)
)

final_allocation["demand_change_abs"] = (
    final_allocation["demand"]
    - final_allocation["prev_month_demand"]
)

final_allocation["demand_change_pct"] = (
    final_allocation["demand_change_abs"]
    / final_allocation["prev_month_demand"].clip(lower=1)
    * 100
)

wide_interval = final_allocation["demand_iqr_ratio"] >= UNCERTAINTY_IQR_RATIO_THRESHOLD
large_jump_pct = final_allocation["demand_change_pct"].abs() >= UNCERTAINTY_CHANGE_PCT_THRESHOLD
large_jump_abs = final_allocation["demand_change_abs"].abs() >= UNCERTAINTY_CHANGE_ABS_THRESHOLD

final_allocation["uncertainty_flag"] = (
    wide_interval
    & large_jump_pct
    & large_jump_abs
).fillna(False)

final_allocation.head()

,pfa_code,pfa_name,msoa_code,msoa_name,month,bucket,capacity_version,demand_q25,demand,demand_q75,...,total_capacity,prev_month_total_alloc,alloc_change_abs,alloc_change_pct,demand_iqr,demand_iqr_ratio,prev_month_demand,demand_change_abs,demand_change_pct,uncertainty_flag
10,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2025-12,acquisitive_crime,local_policing,3.3986,9.6647,20.3897,...,13762.0,NaN,NaN,NaN,16.9911,1.758058,NaN,NaN,NaN,False
13,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2026-01,acquisitive_crime,local_policing,3.3632,9.6296,20.6521,...,13762.0,3.278975,-0.011460,-0.349486,17.2889,1.795391,9.6647,-0.0351,-0.363177,False
16,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2026-02,acquisitive_crime,local_policing,2.9717,9.2404,19.5667,...,13762.0,3.267515,0.016969,0.519330,16.5950,1.795918,9.6296,-0.3892,-4.041705,False
11,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2025-12,disorder_damage,local_policing,19.3513,33.3510,51.4709,...,13762.0,NaN,NaN,NaN,32.1196,0.963078,NaN,NaN,NaN,False
14,E23000001,Metropolitan Police,E02000002,Barking and Dagenham 001,2026-01,disorder_damage,local_policing,19.5435,33.8425,52.4883,...,13762.0,6.956043,-0.028359,-0.407694,32.9448,0.973474,33.3510,0.4915,1.473719,False


In [28]:
final_allocation["uncertainty_flag"].value_counts()


uncertainty_flag
False    61485
True       219
Name: count, dtype: int64

In [29]:
final_allocation.groupby(["pfa_name", "month"])[
    ["baseline_alloc", "variable_alloc", "total_alloc"]
].sum()

baseline_alloc  variable_alloc  total_alloc
pfa_name          month                                               
Avon and Somerset 2025-12           853.5           853.5       1707.0
                  2026-01           853.5           853.5       1707.0
                  2026-02           853.5           853.5       1707.0
Bedfordshire      2025-12           261.5           261.5        523.0
                  2026-01           261.5           261.5        523.0
...                                   ...             ...          ...
West Yorkshire    2026-01          1480.5          1480.5       2961.0
                  2026-02          1480.5          1480.5       2961.0
Wiltshire         2025-12           321.5           321.5        643.0
                  2026-01           321.5           321.5        643.0
                  2026-02           321.5           321.5        643.0

[117 rows x 3 columns]

## Save outputs

In [30]:
OUTPUT_DIR.mkdir(exist_ok=True)

final_output_path = OUTPUT_DIR / "final_allocation_v1.csv"
uncertainty_output_path = OUTPUT_DIR / "uncertainty_flags_v1.csv"
force_summary_output_path = OUTPUT_DIR / "force_msoa_resource_summary_v1.csv"

final_allocation.to_csv(final_output_path, index=False)
final_allocation[final_allocation["uncertainty_flag"]].to_csv(
    uncertainty_output_path,
    index=False,
)
force_msoa_resource_summary.to_csv(force_summary_output_path, index=False)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    final_allocation.to_sql("final_allocation_v1", conn, if_exists="replace", index=False)
    final_allocation[final_allocation["uncertainty_flag"]].to_sql(
        "uncertainty_flags_v1",
        conn,
        if_exists="replace",
        index=False,
    )
    msoa_context.to_sql("msoa_allocation_context_v1", conn, if_exists="replace", index=False)
    force_capacity_model.to_sql("force_capacity_v1", conn, if_exists="replace", index=False)
    bucket_baseline_shares.to_sql("bucket_baseline_shares_v1", conn, if_exists="replace", index=False)
    baseline_allocation.to_sql("baseline_allocation_v1", conn, if_exists="replace", index=False)
    variable_allocation.to_sql("variable_allocation_v1", conn, if_exists="replace", index=False)
    force_bucket_demand.to_sql("force_bucket_demand_v1", conn, if_exists="replace", index=False)
    force_msoa_resource_summary.to_sql("force_msoa_resource_summary_v1", conn, if_exists="replace", index=False)

print("Saved final allocation rows:", len(final_allocation))
print("Final output file:", final_output_path)
print("Uncertainty flags file:", uncertainty_output_path)
print("Force summary file:", force_summary_output_path)
print("Database:", ALLOC_DB_PATH)

Saved final allocation rows: 61704
Final output file: ../outputs/final_allocation_v1.csv
Uncertainty flags file: ../outputs/uncertainty_flags_v1.csv
Force summary file: ../outputs/force_msoa_resource_summary_v1.csv
Database: ../../data/allocation_model.db
